In [1]:
!pip install pillow opencv-python
!pip install python-pptx pillow

  Using cached opencv_python-4.11.0.86-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.11.0.86-cp37-abi3-win_amd64.whl (39.5 MB)
  Using cached python_pptx-1.0.2-py3-none-any.whl.metadata (2.5 kB)
Using cached python_pptx-1.0.2-py3-none-any.whl (472 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------- ----------------------------- 1.0/4.0 MB 5.6 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.0 MB 5.6 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.0 MB 5.8 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 5.7 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 3.7.4.3
    Uninstalling typing-extensions-3.7.4.3:
      Successfully uninstalled typing-extensions-3.7.4.3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-gpu 2.5.0 requires typing-extensions~=3.7.4, but you have typing-extensions 4.13.2 which is incompatible.


# GENERATE IMAGES

In [2]:
from pptx import Presentation
from PIL import Image
import os

def ppt_to_images(ppt_file, output_folder):
    # Load presentation
    prs = Presentation(ppt_file)

    # Create output directory if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Loop through each slide
    for i, slide in enumerate(prs.slides):
        # Create a blank image for each slide (you can set size)
        img = Image.new('RGB', (1920, 1080), color=(255, 255, 255))

        # Placeholder for where content of the slide should go
        # (optional: rendering slide content here is tricky and would require more work)

        # Save each slide as an image
        img_path = os.path.join(output_folder, f'slide_{i + 1}.png')
        img.save(img_path)

    print(f"All slides saved as images in {output_folder}")

# Example usage
#ppt_file = 'your_presentation.pptx'  # Replace with your ppt file path


In [5]:
import random
from PIL import Image, ImageDraw, ImageFont
import textwrap
import os

def generate_image(
    font_path, img_background_color, output_path, text_color, img_size=(1280, 720),
    font_size=40, title_font_size=60, logo_size=(100, 100), line_spacing=10,
    border=False, border_color="black", border_width=10, logo_path=None,
    logo_position="bottom_right", add_image=None, image_position="center", title=None,
    text1=None, text2=None, text3=None, bullet=False
):
    # Create background image
    img = Image.new("RGB", img_size, img_background_color)
    draw = ImageDraw.Draw(img)
    width, height = img_size

    # Add border if needed
    if border:
        draw.rectangle(
            [(border_width // 2, border_width // 2), (width - border_width // 2, height - border_width // 2)],
            outline=border_color, width=border_width
        )

    # Define text area starting positions
    title_y = border_width + 20  # Title starts near the top
    text_x, text_y = border_width + 50, title_y + 80  # Body text starts below the title
    text_area_width = width - 2 * (border_width + 50)

    # Draw title if provided
    if title:
        font_title = ImageFont.truetype(font_path, title_font_size)
        title_bbox = draw.textbbox((0, 0), title, font=font_title)  # Get title bounding box
        title_width = title_bbox[2] - title_bbox[0]  # Width of the title
        title_height = title_bbox[3] - title_bbox[1]  # Height of the title
        title_x = (width - title_width) // 2  # Center horizontally
        title_y = border_width + 20  # Add space from the top
        draw.text((title_x, title_y), title, fill=text_color, font=font_title)
        text_y = title_y + title_height + 30  # Start body text below the title

    # Add bullets to text if required
    if bullet:
        bullet_symbol = "• "
        text1 = bullet_symbol + text1 if text1 else None
        text2 = bullet_symbol + text2 if text2 else None
        text3 = bullet_symbol + text3 if text3 else None

    # Prepare and draw text
    texts = [text1, text2, text3]
    for text in texts:
        if text:
            font = ImageFont.truetype(font_path, font_size)
            wrapped_text = textwrap.fill(text, width=40)
            for line in wrapped_text.split("\n"):
                draw.text((text_x, text_y), line, fill=text_color, font=font)
                text_y += font.getbbox(line)[3] + line_spacing

    # Function to resize and paste images
    def place_image(image_path, position, size):
        image = Image.open(image_path)
        image.thumbnail(size)
        img.paste(image, position, image if image.mode == 'RGBA' else None)

    # Add logo
    if logo_path:
        logo_positions = {
            "top_left": (border_width + 10, border_width + 10),
            "top_right": (width - logo_size[0] - border_width - 10, border_width + 10),
            "bottom_left": (border_width + 10, height - logo_size[1] - border_width - 10),
            "bottom_right": (width - logo_size[0] - border_width - 10, height - logo_size[1] - border_width - 10),
        }
        place_image(logo_path, logo_positions[logo_position], logo_size)

    # Place inline image BELOW the last text block, calculate free space
    if add_image:
        inline_img = Image.open(add_image)
    
        # Calculate the free vertical space after the text block
        free_space = img_size[1] - text_y - 20  # Remaining space below the text, 20 pixels margin
    
        # Calculate new image height (70% of the free space) and resize the image
        image_height = int(free_space * 0.7)  # 70% of the available space below the text
        aspect_ratio = inline_img.width / inline_img.height  # Maintain the aspect ratio
        image_width = int(image_height * aspect_ratio)  # Adjust width based on aspect ratio
    
        # Resize the image to the calculated size
        inline_img = inline_img.resize((image_width, image_height))
    
        if image_position == "left":
            img_x = 10  # Small margin on the left
        elif image_position == "center":
            img_x = (img_size[0] - inline_img.width) // 2  # Center horizontally
        else:  # "right"
            img_x = img_size[0] - inline_img.width - 10  # Small margin on the right
    
        # Vertical position (always 20 pixels below the last text block)
        img_y = text_y + 20  # 20 pixels below the last text block
    
        # Paste the resized image onto the slide
        img.paste(inline_img, (img_x, img_y), inline_img if inline_img.mode == "RGBA" else None)

    # Save the output image
    img.save(output_path)

# Function to construct full file path
def full_path(file_name):
    downloads_path = "../data/"
    return os.path.join(downloads_path, file_name)



In [4]:
import os
import shutil

def delete_all_data_from_dir(directory_path):
    """Deletes all files and subdirectories from the specified directory."""

    # Check if the directory exists
    if os.path.exists(directory_path) and os.path.isdir(directory_path):
        # Iterate over all the files and directories in the specified directory
        for filename in os.listdir(directory_path):
            file_path = os.path.join(directory_path, filename)
            try:
                # If it's a file, remove it
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                # If it's a directory, remove it and its contents recursively
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f"Failed to delete {file_path}. Reason: {e}")
        print(f"All files and subdirectories deleted from {directory_path}")
    else:
        print(f"Directory {directory_path} does not exist or is not accessible.")

# Example usage
#delete_all_data_from_dir(full_path('dataset'))



Directory C:\Users\maria\Downloads\files\dataset does not exist or is not accessible.


In [8]:
import random
import string

def generate_random_text(min_length=5, max_length=40):
    # Randomly select a length for the text within the specified range
    text_length = random.randint(min_length, max_length)

    # Generate a random string of the selected length using uppercase, lowercase letters, and spaces
    random_text = ''.join(random.choices(string.ascii_letters + ' ', k=text_length))

    return random_text

# Example usage
random_text = generate_random_text()
print(f"Generated text of length {len(random_text)}:\n{random_text}")

Generated text of length 18:
cWXUkjbXaZOFpqWeZU


In [9]:
import random
from collections import Counter

def create_balanced_dataset(dataset_directory, num_samples=50):
    """
    Generates a dataset with 50% identical pairs and 50% pairs differing by one attribute,
    with variations in text quantity, images, and border configurations.
    """
    # Placeholder lists for storing results
    image_pairs = []
    style_labels = []
    font_labels = []

    # Target number of samples for each label (0s and 1s)
    target_per_class = num_samples // 2

    # Counters for label balancing
    style_label_count = Counter({0: 0, 1: 0})

    i = 0  # Counter for image naming

    # # Ensure required lists are defined
    # font_list = ["Arial", "Times New Roman", "Verdana", "Helvetica"]
    # background_colors = ["#FFFFFF", "#000000", "#FF5733", "#33FF57"]
    # text_colors = ["#000000", "#FFFFFF", "#333333", "#CCCCCC"]
    # border = [1, 2, 3, 4]
    # border_color = "#000000"
    # logo_path = ["path/to/logo1.png", "path/to/logo2.png"]
    # logo_position = ["top-left", "top-right", "bottom-left", "bottom-right"]
    # inline_img = ["path/to/inline_img1.png", "path/to/inline_img2.png"]

    # Keep generating images until we have exactly `num_samples` pairs
    while len(image_pairs) < num_samples:
        # Decide whether the images will be identical or differ by one attribute
        is_same = random.random() < 0.5

        if is_same and style_label_count[1] < target_per_class:
            # Identical attributes
            font_1 = font_2 = random.choice(font_list)
            bg_color_1 = bg_color_2 = random.choice(background_colors)
            text_color_1 = text_color_2 = random.choice(text_colors)
            border_1 = border_2 = random.choice(border)
            logo_path_1 = logo_path_2 = random.choice(logo_path)
            logo_position1 = logo_position2 = random.choice(logo_position)
            has_bullets_1 = has_bullets_2 = random.choice([True, False])
            style_label = 1
        elif not is_same and style_label_count[0] < target_per_class:
            # Differing attributes
            style_label = 0
            font_1 = random.choice(font_list)
            font_2 = font_1 if random.random() < 0.5 else random.choice(font_list)
            bg_color_1 = random.choice(background_colors)
            bg_color_2 = bg_color_1 if random.random() < 0.5 else random.choice(background_colors)
            text_color_1 = random.choice(text_colors)
            text_color_2 = text_color_1 if random.random() < 0.5 else random.choice(text_colors)
            border_1 = random.choice(border)
            border_2 = border_1 if random.random() < 0.5 else random.choice(border)
            logo_path_1 = random.choice(logo_path)
            logo_path_2 = logo_path_1 if random.random() < 0.5 else random.choice(logo_path)
            logo_position1 = random.choice(logo_position)
            logo_position2 = random.choice(logo_position)
            has_bullets_1 = random.choice([True, False])
            has_bullets_2 = has_bullets_1 if random.random() < 0.5 else random.choice([True, False])
        else:
            continue

        # Generate random text (1-3 lines)
        texts = [generate_random_text() for _ in range(random.randint(1, 3))]
        line_spacing = random.choice([1, 2])
        border_status = random.choice([True, False])

        # Determine add_image value: random choice from inline_img if adding an image
        add_image = random.choice(inline_img) if random.random() < 0.3 else None

        # Generate the two images with specified attributes
        img_1_path = f"{dataset_directory}\\img1_{i}.png"
        img_2_path = f"{dataset_directory}\\img2_{i}.png"
        i += 1

        generate_image(
            font_path=font_1,
            img_background_color=bg_color_1,
            output_path=img_1_path,
            text_color=text_color_1,
            img_size=img_size,
            font_size=random.choice((7, 9, 12)),
            title_font_size=random.choice((18, 20, 24)),
            line_spacing=line_spacing,
            logo_size=logo_size,
            border=border_status,
            border_color=border_color,
            border_width=border_1,
            logo_path=logo_path_1,
            logo_position=logo_position1,
            add_image=add_image,
            title=texts[0] if len(texts) > 0 else None,
            text1=texts[1] if len(texts) > 1 else None,
            text2=texts[2] if len(texts) > 2 else None,
            bullet=has_bullets_1
        )
        generate_image(
            font_path=font_2,
            img_background_color=bg_color_2,
            output_path=img_2_path,
            text_color=text_color_2,
            img_size=img_size,
            font_size=random.choice((7, 9, 12)),
            title_font_size=random.choice((18, 20, 24)),
            line_spacing=line_spacing,
            logo_size= logo_size,
            border=border_status,
            border_color=border_color,
            border_width=border_2,
            logo_path=logo_path_2,
            logo_position=logo_position2,
            add_image=add_image,
            title=texts[0] if len(texts) > 0 else None,
            text1=texts[1] if len(texts) > 1 else None,
            text2=texts[2] if len(texts) > 2 else None,
            bullet=has_bullets_2
        )

        font_label = 1 if font_1 == font_2 else 0

        # Append image paths and labels
        image_pairs.append((img_1_path, img_2_path))
        style_labels.append(style_label)
        font_labels.append(font_label)

        # Update counters
        style_label_count[style_label] += 1

    return image_pairs, style_labels, font_labels


In [10]:
import random
import os
from PIL import Image
import numpy as np

# Define list of fonts, background colors, and text colors
# Define list of fonts, background colors, and text colors
font_list = [
    '../data/arial.ttf',
    '../data/timr45w.ttf',
    '../data/Fredoka-VariableFont_wdth,wght.ttf',
    '../data/Rhuma Sinera Inline.ttf',
    '../data/Rhuma Sinera Regular.ttf',
    '../data/Rhuma Sinera Shadow.ttf',
    '../data/RobotoFlex-Regular.ttf',
    '../data/FirstickDemoRegular.ttf',
    '../data/Ebony Eyes Personal Use.ttf'
]

background_colors = ['white', 'lightgray', 'lightblue', 'gray', 'pink']
text_colors = ['black', 'red', 'blue', 'green', 'brown']

logo_path = [
    '../data/download.png',
    '../data/logo2.jpg'
]

inline_img = [
    '../data/img1.jpg',
    '../data/img2.png',
    '../data/img3.jpg',
    '../data/img4.png',
    '../data/img5.jpg'
]
logo_position=['top_left','top_right','bottom_left','bottom_right']

border_color = 'black'
img_size = (224, 224)
logo_size = (10, 10)
border = [True, False]
img_size=(224, 224)
border=[True,False]
border_width=10
dataset_directory = '../data/dataset1'

# Directory for saving generated images
os.makedirs(dataset_directory, exist_ok=True)
delete_all_data_from_dir(dataset_directory)

image_pairs, style_labels, font_labels = create_balanced_dataset(dataset_directory,num_samples=60)
print("ready")


All files and subdirectories deleted from ../data/dataset1
ready


In [11]:
import pandas as pd

df = pd.DataFrame({
    'img1_path': [pair[0] for pair in image_pairs],
    'img2_path': [pair[1] for pair in image_pairs],
    'style_label': style_labels,
    'font_label': font_labels
})
df.to_csv('../data/image_pairs_labels.csv', index=False)

,img1_path,img2_path,style_label,font_label
0,../data/dataset1\img1_0.png,../data/dataset1\img2_0.png,1,1
1,../data/dataset1\img1_1.png,../data/dataset1\img2_1.png,1,1
2,../data/dataset1\img1_2.png,../data/dataset1\img2_2.png,1,1
3,../data/dataset1\img1_3.png,../data/dataset1\img2_3.png,0,0
4,../data/dataset1\img1_4.png,../data/dataset1\img2_4.png,1,1
5,../data/dataset1\img1_5.png,../data/dataset1\img2_5.png,0,1
6,../data/dataset1\img1_6.png,../data/dataset1\img2_6.png,1,1
7,../data/dataset1\img1_7.png,../data/dataset1\img2_7.png,0,1
8,../data/dataset1\img1_8.png,../data/dataset1\img2_8.png,1,1
9,../data/dataset1\img1_9.png,../data/dataset1\img2_9.png,0,0
